# YOLOv8 Model Compression Demo

**IITP (ETRI Project) - On_Device_AI**

이 노트북에서는 `yolo_utils`를 사용하여 YOLOv8 모델을 압축하는 방법을 보여줍니다.

## 주요 기능
- **Structured Pruning**: L2-norm 기반 필터 제거
- **Knowledge Distillation**: Teacher-Student 학습
- **Channel Reducing**: 물리적 채널 제거

---

## 1. 환경 설정

In [ ]:
# Google Colab에서 실행 시
!pip install ultralytics -q

# 권환 있느 현재 프로젝트 클론 (Colab용)
!git clone 
%cd On_Device_AI

In [ ]:
import sys
import os
import torch
import numpy as np

# yolo_utils 경로 추가 (노트북 위치에 관계없이 동작)
# 메인 폴더(On_Device_AI)에서 실행해도, notebooks 폴더에서 실행해도 동작
if 'yolo_utils' not in sys.modules:
    # 현재 작업 디렉토리 확인
    cwd = os.getcwd()
    if cwd.endswith('notebooks'):
        sys.path.insert(0, os.path.dirname(cwd))
    # 메인 폴더에서 실행 시 추가 설정 불필요

from ultralytics import YOLO

# GPU 확인
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 2. yolo_utils Import

In [ ]:
from yolo_utils import (
    # Pruning
    yolov8_pruning,
    get_filter_pruning_idx,
    
    # Reducing
    yolov8_reducing,
    get_survived_filter_idx,
    
    # Knowledge Distillation
    distillation_loss,
    compute_kd_loss,
    
    # Memory
    model_memory_usage,
    custom_memory_loss_function,
)

print("yolo_utils imported successfully!")

## 3. 모델 로드 및 정보 확인

In [ ]:
# YOLOv8n 모델 로드
model = YOLO('yolov8n.pt')

# 원본 파라미터 수 확인
original_params = sum(p.numel() for p in model.model.parameters())
print(f"Original model parameters: {original_params:,}")

In [ ]:
# 모델 구조 확인
print("\nModel Structure:")
for i, layer in enumerate(model.model.model):
    print(f"  [{i:2d}] {type(layer).__name__}")

## 4. Pruning 적용

L2-norm 기반으로 중요도가 낮은 필터를 0으로 마스킹합니다.

In [ ]:
# Pruning 적용 (30% 필터 제거)
sparsity = 0.3
print(f"Applying pruning with sparsity={sparsity}...")

yolov8_pruning(model.model.model, sparsity=sparsity)

# Pruning 후 0인 파라미터 수 확인
zero_params = 0
with torch.no_grad():
    for p in model.model.parameters():
        zero_params += (p == 0).sum().item()

print(f"\nPruning Results:")
print(f"  Total parameters: {original_params:,}")
print(f"  Zero parameters: {zero_params:,}")
print(f"  Actual sparsity: {zero_params / original_params * 100:.1f}%")

## 5. Pruning된 모델 테스트

In [ ]:
# 테스트 이미지로 추론
results = model.predict(
    source='https://ultralytics.com/images/bus.jpg',
    save=False,
    verbose=False
)

print(f"Detected {len(results[0].boxes)} objects")
for box in results[0].boxes:
    cls = int(box.cls[0])
    conf = float(box.conf[0])
    print(f"  - {results[0].names[cls]}: {conf:.2f}")

## 6. Reducing 적용 (물리적 채널 제거)

0으로 마스킹된 필터를 실제로 제거하여 모델 크기를 줄입니다.

In [ ]:
# 새로운 빈 모델 생성
reduced_model = YOLO('yolov8n.yaml')

# Reducing 적용
print("Applying reducing...")
yolov8_reducing(model.model.model, reduced_model.model.model)

# 결과 확인
reduced_params = sum(p.numel() for p in reduced_model.model.parameters())
print(f"\nReducing Results:")
print(f"  Original parameters: {original_params:,}")
print(f"  Reduced parameters: {reduced_params:,}")
print(f"  Compression ratio: {(1 - reduced_params/original_params)*100:.1f}%")

## 7. 모델 저장

In [ ]:
# Pruned 모델 저장 (0으로 마스킹된 상태)
torch.save(model.model.state_dict(), 'yolov8n_pruned.pt')
print("Saved: yolov8n_pruned.pt")

# Reduced 모델 저장 (물리적으로 축소된 상태)
torch.save(reduced_model.model.state_dict(), 'yolov8n_reduced.pt')
print("Saved: yolov8n_reduced.pt")

## 8. Knowledge Distillation 예시

Teacher 모델의 출력을 Student 모델이 모방하도록 학습합니다.

In [ ]:
# Teacher 모델 (더 큰 모델)
teacher = YOLO('yolov8s.pt')  # 실제로는 yolov8x 권장
teacher.model.eval()
for p in teacher.model.parameters():
    p.requires_grad = False

# Student 모델 (압축 대상)
student = YOLO('yolov8n.pt')

print(f"Teacher params: {sum(p.numel() for p in teacher.model.parameters()):,}")
print(f"Student params: {sum(p.numel() for p in student.model.parameters()):,}")

In [ ]:
# KD Loss 계산 예시
dummy_input = torch.randn(1, 3, 640, 640).to(device)

with torch.no_grad():
    teacher_out = teacher.model(dummy_input)
    
student_out = student.model(dummy_input)

# KD Loss 계산
if isinstance(student_out, tuple):
    student_feat = student_out[0]
    teacher_feat = teacher_out[0]
    
    kd_loss = compute_kd_loss(student_feat, teacher_feat, distill_ratio=0.5)
    print(f"KD Loss: {kd_loss.item():.4f}")

## 9. 메모리 사용량 측정 (GPU 필요)

In [ ]:
if torch.cuda.is_available():
    # 메모리 측정
    x = torch.randn(1, 3, 640, 640)
    
    memory = model_memory_usage(x, model, device)
    print(f"Model memory usage: {memory:.2f} MB")
    
    # 메모리 제약 손실 함수
    target_memory = 50.0  # MB
    mem_loss = custom_memory_loss_function(memory, hyperparam=0.1, device_condition_memory=target_memory)
    print(f"Memory loss (target={target_memory}MB): {mem_loss.item():.4f}")
else:
    print("GPU not available, skipping memory measurement")

## 10. 전체 압축 파이프라인 요약

```python
from yolo_utils import yolov8_pruning, yolov8_reducing
from ultralytics import YOLO
import torch

# 1. 모델 로드
model = YOLO('yolov8n.pt')

# 2. Pruning 적용
yolov8_pruning(model.model.model, sparsity=0.3)

# 3. (선택) 학습 수행
# model.train(data='coco8.yaml', epochs=100)

# 4. Reducing 적용
reduced_model = YOLO('yolov8n.yaml')
yolov8_reducing(model.model.model, reduced_model.model.model)

# 5. 저장
torch.save(reduced_model.model.state_dict(), 'compressed.pt')
```

---

## 참고 자료

- [ultralytics_custom/README.md](../ultralytics_custom/README.md) - 상세 구현 문서
- [yolo_utils/](../yolo_utils/) - 압축 유틸리티 모듈